Feature engineering

In [ ]:
# CNN with Featured data Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)

In [1]:
import h5py
import numpy as np
import pandas as pd

In [ ]:
# -----------------------------
# Config
# -----------------------------
IN_PATH_train = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\norm\nights_train_norm.h5"

IN_PATH_test = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\norm\nights_test_norm.h5"
FS = 100

# chanel names
SIGNALS_NAME = [
    "AbdoBelt",
    "AirFlow",
    "PPG",
    "ThorBelt",
    "Snoring",
    "SPO2",
    "C4A1",
    "O2A1",
]

CH = {name: i for i, name in enumerate(SIGNALS_NAME)}

In [3]:
# -----------------------------
# Helpers (FINAL)
# -----------------------------

def roll_min_center(x_1hz, win):
    return pd.Series(x_1hz).rolling(win, center=True, min_periods=1).min().to_numpy()

def roll_max_center(x_1hz, win):
    return pd.Series(x_1hz).rolling(win, center=True, min_periods=1).max().to_numpy()

def roll_mean_center(x_1hz, win):
    return pd.Series(x_1hz).rolling(win, center=True, min_periods=1).mean().to_numpy()

def roll_sum_center(mask_1hz, win):
    return pd.Series(mask_1hz.astype(np.float32)).rolling(win, center=True, min_periods=1).sum().to_numpy()

def roll_corr_center(a_1hz, b_1hz, win):
    a = pd.Series(a_1hz)
    b = pd.Series(b_1hz)
    return a.rolling(win, center=True, min_periods=3).corr(b).fillna(0.0).to_numpy()

def roll_q_center(x_1hz, win, q):
    return pd.Series(x_1hz).rolling(win, center=True, min_periods=1).quantile(q).to_numpy()



In [4]:
import numpy as np
import pandas as pd

def extract_features_subject(X_subj_100hz_8ch, fs=100):
    T = X_subj_100hz_8ch.shape[0]
    n_sec = T // fs
    Xs = X_subj_100hz_8ch[:n_sec * fs].astype(np.float32)

    air = Xs[:, CH["AirFlow"]].reshape(n_sec, fs)
    abd = Xs[:, CH["AbdoBelt"]].reshape(n_sec, fs)
    tho = Xs[:, CH["ThorBelt"]].reshape(n_sec, fs)
    snr = Xs[:, CH["Snoring"]].reshape(n_sec, fs)
    spo = Xs[:, CH["SPO2"]].reshape(n_sec, fs)

    def rms_centered(x2d):
        xc = x2d - x2d.mean(axis=1, keepdims=True)
        return np.sqrt(np.mean(xc * xc, axis=1) + 1e-12)

    airflow_rms = rms_centered(air)
    abd_rms     = rms_centered(abd)
    thor_rms    = rms_centered(tho)
    snore_rms   = rms_centered(snr)

    airflow_ptp = air.max(axis=1) - air.min(axis=1)

    # SPO2 as proxy (no clinical units)
    spo_proxy_mean = spo.mean(axis=1)

    # -----------------------------
    # Airflow validity mask
    # -----------------------------
    ptp_thr = np.percentile(airflow_ptp, 20)
    rms_thr = np.percentile(airflow_rms, 10)
    valid_air = (airflow_ptp > ptp_thr) & (airflow_rms > rms_thr)

    # Protect airflow series for rolling quantile (ignore invalid seconds)
    airflow_rms_safe = airflow_rms.copy()
    airflow_rms_safe[~valid_air] = np.nan

    # -----------------------------
    # DEBUG (rodar uma vez ou para 1 sujeito)
    # -----------------------------
    print("valid_air rate:", valid_air.mean())
    print("NaN rate airflow_rms_safe:", np.isnan(airflow_rms_safe).mean())


    # -----------------------------
    # Local baseline (offline): 5 min window (300s)
    # -----------------------------
    baseline_local = (
        pd.Series(airflow_rms_safe)
        .rolling(300, center=True, min_periods=60)
        .quantile(0.70)
        .to_numpy()
    )

    # Fill edges / all-NaN segments with global robust fallback
    fallback_baseline = np.nanpercentile(airflow_rms_safe, 70)
    baseline_local = np.where(np.isfinite(baseline_local), baseline_local, fallback_baseline)
    baseline_local = np.maximum(baseline_local, 1e-6).astype(np.float32)

    # -----------------------------
    # Airflow drop: use rolling quantile (30s) on safe signal
    # -----------------------------
    low_air_c30 = (
        pd.Series(airflow_rms_safe)
        .rolling(30, center=True, min_periods=1)
        .quantile(0.30)
        .to_numpy()
    )
    # Fill NaNs (windows where everything was invalid) with a small safe value
    low_fallback = np.nanpercentile(airflow_rms_safe, 30)
    low_air_c30 = np.where(np.isfinite(low_air_c30), low_air_c30, low_fallback)
    low_air_c30 = np.maximum(low_air_c30, 0.0).astype(np.float32)

    airflow_drop_pct_c30 = 1.0 - (low_air_c30 / (baseline_local + 1e-12))
    airflow_drop_pct_c30 = np.clip(airflow_drop_pct_c30, 0.0, 1.0).astype(np.float32)

    # Durations below thresholds within 30s centered window
    air_use = np.where(valid_air, airflow_rms, np.nan)
    below30 = np.isfinite(air_use) & (air_use < (0.70 * baseline_local))
    below10 = np.isfinite(air_use) & (air_use < (0.10 * baseline_local))


    airflow_time_below30_c30 = roll_sum_center(below30, 30).astype(np.float32)
    airflow_time_below10_c30 = roll_sum_center(below10, 30).astype(np.float32)

    if np.random.rand() < 0.001:  # imprime raramente
        print("baseline_local q:", np.percentile(baseline_local, [50,90,99]))
        print("low_air_c30 q:", np.percentile(low_air_c30, [50,90,99]))


    # -----------------------------
    # Snore
    # -----------------------------
    snore_thr = float(np.percentile(snore_rms, 90))
    snore_high = snore_rms > snore_thr
    snore_presence_c30 = (roll_sum_center(snore_high, 30) / 30.0).astype(np.float32)

    # -----------------------------
    # Effort
    # -----------------------------
    abd_rms_c10  = roll_mean_center(abd_rms, 10)
    thor_rms_c10 = roll_mean_center(thor_rms, 10)
    effort_sum_c10 = (abd_rms_c10 + thor_rms_c10).astype(np.float32)

    thor_abd_corr_c30 = roll_corr_center(thor_rms, abd_rms, 30).astype(np.float32)

    # -----------------------------
    # SPO2 PROXY (robust normalization per subject)
    # -----------------------------
    med = np.median(spo_proxy_mean)
    mad = np.median(np.abs(spo_proxy_mean - med)) + 1e-6
    spo_norm = (spo_proxy_mean - med) / (1.4826 * mad)

    spo_min_c30   = roll_min_center(spo_norm, 30).astype(np.float32)
    spo_max_c30   = roll_max_center(spo_norm, 30).astype(np.float32)
    spo_range_c30 = (spo_max_c30 - spo_min_c30).astype(np.float32)

    spo_slope_1s  = np.gradient(spo_norm)
    spo_slope_c30 = roll_mean_center(spo_slope_1s, 30).astype(np.float32)

    # -----------------------------
    # Composites
    # -----------------------------
    airflow_drop_x_spo_range = (airflow_drop_pct_c30 * spo_range_c30).astype(np.float32)
    airflow_lowdur_x_spo_rng = (airflow_time_below30_c30 * spo_range_c30).astype(np.float32)

    feats = {
        "airflow_rms_1s": airflow_rms.astype(np.float32),
        "airflow_ptp_1s": airflow_ptp.astype(np.float32),
        "abd_rms_1s": abd_rms.astype(np.float32),
        "thor_rms_1s": thor_rms.astype(np.float32),
        "snore_rms_1s": snore_rms.astype(np.float32),
        "spo_proxy_mean_1s": spo_proxy_mean.astype(np.float32),

        # local baseline (better than constant)
        "airflow_baseline_local": baseline_local.astype(np.float32),

        "airflow_drop_pct_c30": airflow_drop_pct_c30,
        "airflow_time_below30_c30": airflow_time_below30_c30,
        "airflow_time_below10_c30": airflow_time_below10_c30,

        "effort_sum_c10": effort_sum_c10,
        "thor_abd_corr_c30": thor_abd_corr_c30,
        "snore_presence_c30": snore_presence_c30,

        "spo_min_c30": spo_min_c30,
        "spo_range_c30": spo_range_c30,
        "spo_slope_c30": spo_slope_c30,

        "airflow_drop_x_spo_range": airflow_drop_x_spo_range,
        "airflow_lowdur_x_spo_rng": airflow_lowdur_x_spo_rng,
    }
    return feats




In [11]:
FS = 100

# -------- Load train
with h5py.File(IN_PATH_train, "r") as f:
    X_train = f["X_nights"][:]      # (n_subj_train, 1800000, 8)
    y_train = f["y_nights"][:]      # (n_subj_train, 18000)
    subj_train = f["subject_ids"][:]

print("X_train:", X_train.shape, "y_train:", y_train.shape, "subj_train:", subj_train.shape)

# -------- Load test
with h5py.File(IN_PATH_test, "r") as f:
    X_test = f["X_nights"][:].astype("float32")
    subj_test = f["subject_ids"][:].astype(int)

print("Xn_test_nights:", X_test.shape)     # (22, 1800000, 8)
print("subj_order_test:", subj_test[:10], "...", subj_test[-10:])


print("X_test:", X_test.shape, "subj_test:", subj_test.shape)

# -------- Feature extraction helper
def build_X_feat(X_nights, fs=100):
    feature_names = None
    X_feat_list = []

    for i in range(X_nights.shape[0]):
        feats = extract_features_subject(X_nights[i], fs=fs)

        if feature_names is None:
            feature_names = list(feats.keys())

        F = np.stack([feats[name] for name in feature_names], axis=1).astype(np.float32)
        X_feat_list.append(F)

    X_feat = np.stack(X_feat_list, axis=0)  # (n_subj, 18000, n_feat)
    return X_feat, feature_names

# -------- Build train features
X_feat_train, feature_names = build_X_feat(X_train, fs=FS)
print("X_feat_train:", X_feat_train.shape, "n_feat:", X_feat_train.shape[-1])

assert X_feat_train.shape[1] == y_train.shape[1], "Train features e labels não alinhados!"

# -------- Build test features (usa os mesmos feature_names implicitamente)
X_feat_test, feature_names_test = build_X_feat(X_test, fs=FS)
print("X_feat_test:", X_feat_test.shape)

# checa se ordem/nomes bateram
assert feature_names_test == feature_names, "Feature names diferentes entre train e test (não deveria)."


X_train: (22, 1800000, 8) y_train: (22, 18000) subj_train: (22,)
Xn_test_nights: (22, 1800000, 8)
subj_order_test: [0 1 2 3 4 5 6 7 8 9] ... [12 13 14 15 16 17 18 19 20 21]
X_test: (22, 1800000, 8) subj_test: (22,)
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.7997222222222222
NaN rate airflow_rms_safe: 0.20027777777777778
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.8
NaN rate airflow_rms_safe: 0.2
valid_air rate: 0.7999444444444445
NaN rate airflow_rms_safe: 0.20005555555555554
valid_air

In [7]:
OUT_TRAIN = IN_PATH_train.replace(".h5", "_features_1hz.h5")

with h5py.File(OUT_TRAIN, "w") as f:
    f.create_dataset("X_feat", data=X_feat_train, compression="gzip")
    f.create_dataset("y_nights", data=y_train, compression="gzip")
    f.create_dataset("subject_ids", data=subj_train)
    f.create_dataset("feature_names", data=np.array(feature_names, dtype="S"))

print("Saved TRAIN features:", OUT_TRAIN)


Saved TRAIN features: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\nights_train_features_1hz.h5


In [8]:
OUT_TEST = IN_PATH_test.replace(".h5", "_features_1hz.h5")

with h5py.File(OUT_TEST, "w") as f:
    f.create_dataset("X_feat", data=X_feat_test, compression="gzip")
    f.create_dataset("subject_ids", data=subj_test)
    f.create_dataset("feature_names", data=np.array(feature_names, dtype="S"))

print("Saved TEST features:", OUT_TEST)


Saved TEST features: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\nights_test_features_1hz.h5


In [13]:
# ========================================
# FLATTEN TRAIN DATA
# ========================================

# X_feat_train: (22, 18000, 18)
# y_train:      (22, 18000)
# subj_train:   (22,)

n_subj, n_sec, n_feat = X_feat_train.shape

# achata
X_flat_train = X_feat_train.reshape(-1, n_feat)
y_flat_train = y_train.reshape(-1)

subjects_flat_train = np.repeat(subj_train, n_sec)

df_train = pd.DataFrame(X_flat_train, columns=feature_names)
df_train["y"] = y_flat_train
df_train["subject"] = subjects_flat_train

print("Train DataFrame shape:", df_train.shape)
print("Train DataFrame head:")
print(df_train.head())

print("\nClass distribution:")
print(df_train["y"].value_counts(normalize=True))

print("\nMissing values:")
print(df_train.isna().sum().sum())

print("\nInfinite values:")
print(np.isinf(df_train[feature_names].values).sum())

Train DataFrame shape: (396000, 20)
Train DataFrame head:
   airflow_rms_1s  airflow_ptp_1s  abd_rms_1s  thor_rms_1s  snore_rms_1s  \
0        0.008695        0.036217    0.008728     0.008668      0.008693   
1        0.009129        0.037479    0.009176     0.009032      0.009038   
2        0.015983        0.063825    0.015960     0.016056      0.016035   
3        0.011031        0.044721    0.011007     0.011069      0.011066   
4        0.022912        0.063830    0.022900     0.023055      0.023153   

   spo_proxy_mean_1s  airflow_baseline_local  airflow_drop_pct_c30  \
0          -0.377602                0.027604              0.680286   
1          -0.377332                0.027604              0.689567   
2          -0.378005                0.027604              0.706407   
3          -0.376184                0.027575              0.776178   
4          -0.380635                0.027547              0.775949   

   airflow_time_below30_c30  airflow_time_below10_c30  effort_su

In [14]:
# ========================================
# FEATURE STATISTICS
# ========================================

print("\n" + "="*70)
print("📊 ESTATÍSTICAS DOS FEATURES")
print("="*70)

print("\nDescrição geral:")
print(df_train[feature_names].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

print("\nVariância por feature:")
feat_var = df_train[feature_names].var().sort_values()
print(feat_var)

print("\nTaxa de valores NaN por feature:")
nan_rate = df_train[feature_names].isna().mean().sort_values(ascending=False)
print(nan_rate[nan_rate > 0])

print("\nTaxa de valores infinitos por feature:")
inf_rate = (np.isinf(df_train[feature_names].values)).sum(axis=0) / len(df_train)
inf_rate_series = pd.Series(inf_rate, index=feature_names).sort_values(ascending=False)
print(inf_rate_series[inf_rate_series > 0])

# ========================================
# DISTRIBUIÇÃO POR SUJEITO
# ========================================

print("\n" + "="*70)
print("👥 DISTRIBUIÇÃO POR SUJEITO")
print("="*70)

subject_stats = df_train.groupby("subject").agg({
    "y": ["sum", "mean", "count"],
}).round(4)
subject_stats.columns = ["Apnea_Seconds", "Apnea_Rate", "Total_Seconds"]
print(subject_stats)

# ========================================
# CORRETUDE: Verificar se os dados estão íntegros
# ========================================

print("\n" + "="*70)
print("✅ VERIFICAÇÕES")
print("="*70)

print(f"\n1. Shape correto?")
print(f"   X_flat_train: {X_flat_train.shape} (esperado: ({n_subj * n_sec}, {n_feat}))")
print(f"   y_flat_train: {y_flat_train.shape} (esperado: ({n_subj * n_sec},))")

print(f"\n2. Labels únicos em y_flat_train: {np.unique(y_flat_train)}")

print(f"\n3. Sujeitos únicos em subjects_flat_train: {np.unique(subjects_flat_train)}")

print(f"\n4. Contagem de samples por sujeito:")
print(np.bincount(subjects_flat_train.astype(int)))


📊 ESTATÍSTICAS DOS FEATURES

Descrição geral:
                             count          mean           std           min  \
airflow_rms_1s            396000.0      0.042483      0.214119  1.000000e-06   
airflow_ptp_1s            396000.0      0.103654      0.465336  0.000000e+00   
abd_rms_1s                396000.0      0.042484      0.214122  1.000000e-06   
thor_rms_1s               396000.0      0.042483      0.214116  1.000000e-06   
snore_rms_1s              396000.0      0.042483      0.214116  1.000000e-06   
spo_proxy_mean_1s         396000.0     -0.000048      0.975870 -4.309928e-01   
airflow_baseline_local    396000.0      0.014874      0.005900  5.854287e-04   
airflow_drop_pct_c30      396000.0      0.771778      0.310059  0.000000e+00   
airflow_time_below30_c30  396000.0     14.845250      7.242309  0.000000e+00   
airflow_time_below10_c30  396000.0      9.251414      7.151683  0.000000e+00   
effort_sum_c10            396000.0      0.084967      0.131163  2.000000e

In [ ]:
df["y"].value_counts(normalize=True)


In [ ]:
df.isna().mean().sort_values(ascending=False)
np.isinf(df[feature_names]).mean().sort_values(ascending=False)


In [15]:
df[feature_names].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T


NameError: name 'df' is not defined

In [ ]:
df[feature_names].var().sort_values()


In [ ]:
with h5py.File(IN_PATH, "r") as f:
    X = f["X_nights"][:]   # (22, T, 8)

# escolhe um sujeito
i = 0

channels = {
    0: "Abdominal",
    1: "Airflow",
    2: "PPG",
    3: "Thoracic",
    4: "Snore",
    5: "SpO2",
    6: "C4-A1",
    7: "O2-A1"
}

for ch, name in channels.items():
    sig = X[i, :, ch]
    print(f"\n{name}")
    print("  min:", np.min(sig))
    print("  max:", np.max(sig))
    print("  mean:", np.mean(sig))
    print("  std:", np.std(sig))


In [ ]:
# achata só pra olhar distribuições
df_check = pd.DataFrame(
    X_feat.reshape(-1, X_feat.shape[-1]),
    columns=feature_names
)

print(df_check[["spo_proxy_mean_1s","spo_min_c30","spo_range_c30"]].describe())


In [16]:
# ========================================
# SAVE TRAIN FEATURES
# ========================================
OUT_TRAIN = IN_PATH_train.replace(".h5", "_features_1hz.h5")

with h5py.File(OUT_TRAIN, "w") as f:
    f.create_dataset("X_feat", data=X_feat_train, compression="gzip")
    f.create_dataset("y_nights", data=y_train, compression="gzip")
    f.create_dataset("subject_ids", data=subj_train)
    f.create_dataset("feature_names", data=np.array(feature_names, dtype="S"))

print("✅ Saved TRAIN features:", OUT_TRAIN)

# ========================================
# SAVE TEST FEATURES
# ========================================
OUT_TEST = IN_PATH_test.replace(".h5", "_features_1hz.h5")

with h5py.File(OUT_TEST, "w") as f:
    f.create_dataset("X_feat", data=X_feat_test, compression="gzip")
    f.create_dataset("subject_ids", data=subj_test)
    f.create_dataset("feature_names", data=np.array(feature_names, dtype="S"))

print("✅ Saved TEST features:", OUT_TEST)

# ========================================
# VERIFY SAVED FILES
# ========================================
print("\n" + "="*70)
print("📋 VERIFICAÇÃO DOS ARQUIVOS SALVOS")
print("="*70)

with h5py.File(OUT_TRAIN, "r") as f:
    print("\n🔴 TRAIN:")
    print(f"  • X_feat: {f['X_feat'].shape}")
    print(f"  • y_nights: {f['y_nights'].shape}")
    print(f"  • subject_ids: {f['subject_ids'].shape}")
    print(f"  • feature_names: {len(f['feature_names'][:])} features")

with h5py.File(OUT_TEST, "r") as f:
    print("\n🟢 TEST:")
    print(f"  • X_feat: {f['X_feat'].shape}")
    print(f"  • subject_ids: {f['subject_ids'].shape}")
    print(f"  • feature_names: {len(f['feature_names'][:])} features")


✅ Saved TRAIN features: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\norm\nights_train_norm_features_1hz.h5
✅ Saved TEST features: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\norm\nights_test_norm_features_1hz.h5

📋 VERIFICAÇÃO DOS ARQUIVOS SALVOS

🔴 TRAIN:
  • X_feat: (22, 18000, 18)
  • y_nights: (22, 18000)
  • subject_ids: (22,)
  • feature_names: 18 features

🟢 TEST:
  • X_feat: (22, 18000, 18)
  • subject_ids: (22,)
  • feature_names: 18 features
